# Notebook 05 — Construcción de `silver.ventas_minoristas`

## Objetivo

Construir la tabla `silver.ventas_minoristas` a partir de `bronze.ventas_minoristas` aplicando los siguientes criterios:

1. **Ventana temporal**: filtrar el rango 2022-01-01 a 2025-12-31, eliminando los años incompletos (2019, 2020, 2021, 2026).
2. **Tipos de datos**: castear `id_cliente` a VARCHAR para garantizar coherencia con `silver.dim_cliente` y `silver.fact_lineas_pedido`.
3. **Identificador de producto**: crear `id_sku` derivado de la concatenación de modelo, color y talla.
4. **Normalización de texto**: aplicar `UPPER(TRIM(...))` a los códigos identificativos y `TRIM(...)` a las descripciones.
5. **Flag de devolución**: incorporar el campo booleano `es_devolucion` calculado a partir del catálogo de códigos de tipo de pedido.
6. **Incorporación de importes**: integrar las columnas económicas `importe_total` e `importe_total_con_descuento` recibidas en la actualización del sistema ERP, junto con el campo derivado `descuento_aplicado` (diferencia entre el importe bruto y el neto).

Este notebook constituye, junto con `silver.fact_lineas_pedido`, una de las dos fuentes operativas de la tabla `gold.cliente_360`, y aporta específicamente la información del canal retail (operaciones de venta a consumidor final) tanto a nivel operativo (volumen, frecuencia, devoluciones) como económico (facturación, descuentos, ticket medio).

## 1. Configuración y conexión a DuckDB

In [29]:
import duckdb
from pathlib import Path

RUTA_PROYECTO = Path.home() / "OneDrive" / "Documentos" / "TFG_Selmark"
RUTA_DUCKDB = RUTA_PROYECTO / "duckdb" / "selmark.duckdb"

con = duckdb.connect(str(RUTA_DUCKDB))
print(f"Conectado a: {RUTA_DUCKDB}")
print(f"Tamaño actual: {RUTA_DUCKDB.stat().st_size / (1024*1024):.2f} MB")

Conectado a: C:\Users\lopec\OneDrive\Documentos\TFG_Selmark\duckdb\selmark.duckdb
Tamaño actual: 308.26 MB


## 2. Inspección previa de `bronze.ventas_minoristas`

Antes de construir Silver, revisamos el estado de partida: volumen, esquema, distribución temporal, canales y calidad de fechas.

In [30]:
print("=" * 60)
print("ESQUEMA DE bronze.ventas_minoristas")
print("=" * 60)
esquema = con.execute("DESCRIBE bronze.ventas_minoristas").fetchdf()
print(esquema.to_string(index=False))

print("\n" + "=" * 60)
print("VOLUMEN TOTAL")
print("=" * 60)
total = con.execute("SELECT COUNT(*) FROM bronze.ventas_minoristas").fetchone()[0]
print(f"Filas totales: {total:,}")

ESQUEMA DE bronze.ventas_minoristas
                     column_name column_type null  key default extra
                     fecha_venta        DATE  YES None    None  None
                      cod_modelo     VARCHAR  YES None    None  None
                     desc_modelo     VARCHAR  YES None    None  None
                       cod_color     VARCHAR  YES None    None  None
                      desc_color     VARCHAR  YES None    None  None
                       cod_serie     VARCHAR  YES None    None  None
                      desc_serie     VARCHAR  YES None    None  None
                           talla     VARCHAR  YES None    None  None
                   cantidad_neta      BIGINT  YES None    None  None
cantidad_ventas_sin_devoluciones      BIGINT  YES None    None  None
                   importe_total      DOUBLE  YES None    None  None
     importe_total_con_descuento      DOUBLE  YES None    None  None
                      id_cliente      BIGINT  YES None    None  Non

In [31]:
print("=" * 60)
print("DISTRIBUCIÓN POR AÑO")
print("=" * 60)
por_anio = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes_distintos,
        MIN(fecha_venta) AS fecha_min,
        MAX(fecha_venta) AS fecha_max
    FROM bronze.ventas_minoristas
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(por_anio.to_string(index=False))

DISTRIBUCIÓN POR AÑO
 anio  filas  clientes_distintos  fecha_min  fecha_max
 2019   7299                 257 2019-09-11 2019-12-11
 2020 179194                1476 2020-01-04 2020-12-31
 2021 703622                2271 2021-01-01 2021-12-31
 2022 729665                2266 2022-01-01 2022-12-31
 2023 719482                2232 2023-01-01 2023-12-31
 2024 705208                2200 2024-01-01 2024-12-31
 2025 713210                2117 2025-01-01 2025-12-31
 2026 251542                1775 2026-01-01 2026-05-21


In [32]:
print("=" * 60)
print("LISTADO COMPLETO DE COLUMNAS DE bronze.ventas_minoristas")
print("=" * 60)
columnas = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'bronze' 
      AND table_name = 'ventas_minoristas'
    ORDER BY ordinal_position
""").fetchdf()
print(columnas.to_string(index=False))

LISTADO COMPLETO DE COLUMNAS DE bronze.ventas_minoristas
                     column_name data_type
                     fecha_venta      DATE
                      cod_modelo   VARCHAR
                     desc_modelo   VARCHAR
                       cod_color   VARCHAR
                      desc_color   VARCHAR
                       cod_serie   VARCHAR
                      desc_serie   VARCHAR
                           talla   VARCHAR
                   cantidad_neta    BIGINT
cantidad_ventas_sin_devoluciones    BIGINT
                   importe_total    DOUBLE
     importe_total_con_descuento    DOUBLE
                      id_cliente    BIGINT
                  nombre_cliente   VARCHAR
                  id_tipo_pedido   VARCHAR
                desc_tipo_pedido   VARCHAR
                    id_temporada    BIGINT


In [33]:
print("=" * 60)
print("DISTRIBUCIÓN POR TIPO DE PEDIDO (CANAL)")
print("=" * 60)
por_tipo = con.execute("""
    SELECT 
        id_tipo_pedido,
        desc_tipo_pedido,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes_distintos
    FROM bronze.ventas_minoristas
    GROUP BY id_tipo_pedido, desc_tipo_pedido
    ORDER BY filas DESC
""").fetchdf()
print(por_tipo.to_string(index=False))

DISTRIBUCIÓN POR TIPO DE PEDIDO (CANAL)
id_tipo_pedido   desc_tipo_pedido   filas  clientes_distintos
             1          Temporada 2359929                2878
             2         Repetición  832730                2887
             7           Depósito  184842                   8
          eCom          Ecommerce  148376                   7
         77/88                ECI  103123                   8
           B2B                B2B   98797                 495
             5           Muestras   77346                 152
             3         Devolución   71866                1743
        D77/88           Dev. ECI   39105                   6
            12      Dev. Muestras   38946                  84
            10      Dev. Depósito   27201                   3
          eDev     Dev. Ecommerce   24371                  15
          RECI Regularización ECI    1498                   1
           PER         PERSONALES     581                  47
            11   Consumo Depós

In [34]:
print("=" * 60)
print("CALIDAD DE CAMPOS CLAVE")
print("=" * 60)
calidad = con.execute("""
    SELECT 
        SUM(CASE WHEN id_cliente IS NULL THEN 1 ELSE 0 END) AS nulos_id_cliente,
        SUM(CASE WHEN fecha_venta IS NULL THEN 1 ELSE 0 END) AS nulos_fecha,
        SUM(CASE WHEN cod_modelo IS NULL OR TRIM(cod_modelo) = '' THEN 1 ELSE 0 END) AS nulos_cod_modelo,
        SUM(CASE WHEN cod_color  IS NULL OR TRIM(cod_color)  = '' THEN 1 ELSE 0 END) AS nulos_cod_color,
        SUM(CASE WHEN cod_serie  IS NULL OR TRIM(cod_serie)  = '' THEN 1 ELSE 0 END) AS nulos_cod_serie,
        SUM(CASE WHEN talla      IS NULL OR TRIM(talla)      = '' THEN 1 ELSE 0 END) AS nulos_talla,
        SUM(CASE WHEN cantidad_neta IS NULL THEN 1 ELSE 0 END) AS nulos_cant_neta,
        SUM(CASE WHEN id_tipo_pedido IS NULL THEN 1 ELSE 0 END) AS nulos_tipo_pedido
    FROM bronze.ventas_minoristas
""").fetchdf()
print(calidad.to_string(index=False))

CALIDAD DE CAMPOS CLAVE
 nulos_id_cliente  nulos_fecha  nulos_cod_modelo  nulos_cod_color  nulos_cod_serie  nulos_talla  nulos_cant_neta  nulos_tipo_pedido
              0.0          0.0               0.0              0.0              0.0          0.0              0.0                0.0


In [35]:
print("=" * 60)
print("ANÁLISIS DE UNICIDAD DEL SKU")
print("=" * 60)

# Opción A: SKU = modelo + color + talla
opt_a = con.execute("""
    SELECT COUNT(DISTINCT (cod_modelo || '-' || cod_color || '-' || talla)) AS combinaciones
    FROM bronze.ventas_minoristas
""").fetchone()[0]

# Opción B: SKU = modelo + color + talla + serie
opt_b = con.execute("""
    SELECT COUNT(DISTINCT (cod_modelo || '-' || cod_color || '-' || talla || '-' || cod_serie)) AS combinaciones
    FROM bronze.ventas_minoristas
""").fetchone()[0]

print(f"Opción A (modelo+color+talla)        : {opt_a:,} SKUs distintos")
print(f"Opción B (modelo+color+talla+serie)  : {opt_b:,} SKUs distintos")
print(f"Diferencia                           : {opt_b - opt_a:,}")

# Detectar combos modelo+color+talla con varias series (si los hay)
print("\n" + "=" * 60)
print("¿La misma combinación modelo+color+talla aparece con varias series?")
print("=" * 60)
multi_serie = con.execute("""
    SELECT COUNT(*) AS combos_con_varias_series
    FROM (
        SELECT cod_modelo, cod_color, talla, COUNT(DISTINCT cod_serie) AS n_series
        FROM bronze.ventas_minoristas
        GROUP BY cod_modelo, cod_color, talla
        HAVING COUNT(DISTINCT cod_serie) > 1
    )
""").fetchone()[0]
print(f"Combos modelo+color+talla con >1 serie: {multi_serie:,}")

ANÁLISIS DE UNICIDAD DEL SKU
Opción A (modelo+color+talla)        : 56,118 SKUs distintos
Opción B (modelo+color+talla+serie)  : 56,118 SKUs distintos
Diferencia                           : 0

¿La misma combinación modelo+color+talla aparece con varias series?
Combos modelo+color+talla con >1 serie: 0


In [36]:
print("=" * 60)
print("DISTRIBUCIÓN DE CANTIDAD_NETA POR TIPO DE PEDIDO")
print("=" * 60)
signo_cant = con.execute("""
    SELECT 
        id_tipo_pedido,
        desc_tipo_pedido,
        SUM(CASE WHEN cantidad_neta > 0 THEN 1 ELSE 0 END) AS filas_positivas,
        SUM(CASE WHEN cantidad_neta < 0 THEN 1 ELSE 0 END) AS filas_negativas,
        SUM(CASE WHEN cantidad_neta = 0 THEN 1 ELSE 0 END) AS filas_cero,
        SUM(cantidad_neta) AS suma_total
    FROM bronze.ventas_minoristas
    GROUP BY id_tipo_pedido, desc_tipo_pedido
    ORDER BY suma_total DESC
""").fetchdf()
print(signo_cant.to_string(index=False))

DISTRIBUCIÓN DE CANTIDAD_NETA POR TIPO DE PEDIDO
id_tipo_pedido   desc_tipo_pedido  filas_positivas  filas_negativas  filas_cero  suma_total
             1          Temporada        2359929.0              0.0         0.0   3802859.0
             2         Repetición         832662.0             68.0         0.0   1737948.0
             7           Depósito         184837.0              5.0         0.0    713757.0
          eCom          Ecommerce         148373.0              1.0         2.0    153210.0
           B2B                B2B          98797.0              0.0         0.0    128356.0
         77/88                ECI         103123.0              0.0         0.0    104878.0
             5           Muestras          77187.0            159.0         0.0     83851.0
          RECI Regularización ECI           1498.0              0.0         0.0     11975.0
           N/A                N/A            139.0             13.0         0.0      1017.0
           PER         PERSONAL

## 2.bis Análisis de las nuevas columnas de importe

La versión actual de `bronze.ventas_minoristas` incorpora dos columnas económicas (`importe_total` e `importe_total_con_descuento`) que no estaban informadas en la primera versión del conjunto de datos. Ambas se reciben ya tipadas como DOUBLE desde el origen, con valores numéricos válidos.

Antes de incorporarlas a la capa Silver, se realiza un análisis exhaustivo de su contenido para confirmar la calidad de los datos y orientar el diseño del esquema final. Las dimensiones que se analizan son las siguientes:

1. **Cobertura general**: cuántas filas tienen valor informado y cuántas presentan valores nulos.
2. **Estadísticas descriptivas**: rango, media, mediana, recuento de negativos y de ceros para ambas columnas, lo que permite caracterizar la distribución y detectar posibles anomalías.
3. **Cobertura por año**: verificar que el relleno es homogéneo a lo largo del periodo 2022-2025, sin sesgos temporales.
4. **Cobertura por canal**: comprobar que todos los canales de venta están informados.
5. **Comparativa entre las dos columnas**: cuantificar las filas en las que se ha aplicado descuento real, las filas en las que ambas columnas coinciden y las filas con inconsistencias residuales del ERP.

In [37]:
print("=" * 60)
print("ANÁLISIS DE LAS NUEVAS COLUMNAS DE IMPORTE")
print("=" * 60)

# 1. Cobertura general sobre todo el bronze
print("\n--- 1. COBERTURA GENERAL (sobre el bronze completo) ---")
cobertura = con.execute("""
    SELECT
        COUNT(*)                                                                   AS total_filas,
        SUM(CASE WHEN importe_total IS NULL THEN 1 ELSE 0 END)                     AS imp_total_nulos,
        SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END)                 AS imp_total_validos,
        SUM(CASE WHEN importe_total_con_descuento IS NULL THEN 1 ELSE 0 END)       AS imp_desc_nulos,
        SUM(CASE WHEN importe_total_con_descuento IS NOT NULL THEN 1 ELSE 0 END)   AS imp_desc_validos
    FROM bronze.ventas_minoristas
""").fetchdf()
print(cobertura.T)

# 2. Estadísticas descriptivas de los importes válidos
print("\n--- 2. ESTADÍSTICAS DESCRIPTIVAS (filas no nulas) ---")
stats = con.execute("""
    SELECT
        'importe_total'                AS columna,
        COUNT(importe_total)           AS no_nulos,
        ROUND(MIN(importe_total), 2)   AS minimo,
        ROUND(AVG(importe_total), 2)   AS media,
        ROUND(MEDIAN(importe_total), 2) AS mediana,
        ROUND(MAX(importe_total), 2)   AS maximo,
        SUM(CASE WHEN importe_total < 0 THEN 1 ELSE 0 END) AS negativos,
        SUM(CASE WHEN importe_total = 0 THEN 1 ELSE 0 END) AS ceros
    FROM bronze.ventas_minoristas
    WHERE importe_total IS NOT NULL
    UNION ALL
    SELECT
        'importe_total_con_descuento',
        COUNT(importe_total_con_descuento),
        ROUND(MIN(importe_total_con_descuento), 2),
        ROUND(AVG(importe_total_con_descuento), 2),
        ROUND(MEDIAN(importe_total_con_descuento), 2),
        ROUND(MAX(importe_total_con_descuento), 2),
        SUM(CASE WHEN importe_total_con_descuento < 0 THEN 1 ELSE 0 END),
        SUM(CASE WHEN importe_total_con_descuento = 0 THEN 1 ELSE 0 END)
    FROM bronze.ventas_minoristas
    WHERE importe_total_con_descuento IS NOT NULL
""").fetchdf()
print(stats.to_string(index=False))

# 3. Cobertura por año dentro de la ventana 2022-2025
print("\n--- 3. COBERTURA POR AÑO (ventana 2022-2025) ---")
por_anio = con.execute("""
    SELECT
        EXTRACT(YEAR FROM fecha_venta)                                                AS anio,
        COUNT(*)                                                                       AS filas,
        SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END)                     AS con_importe,
        ROUND(100.0 * SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END) 
              / COUNT(*), 2)                                                           AS pct_relleno
    FROM bronze.ventas_minoristas
    WHERE fecha_venta BETWEEN DATE '2022-01-01' AND DATE '2025-12-31'
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(por_anio.to_string(index=False))

# 4. Cobertura por canal (id_tipo_pedido) en la ventana del proyecto
print("\n--- 4. COBERTURA POR CANAL (id_tipo_pedido) ---")
por_canal = con.execute("""
    SELECT
        id_tipo_pedido,
        desc_tipo_pedido,
        COUNT(*)                                                                       AS filas,
        SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END)                     AS con_importe,
        ROUND(100.0 * SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END) 
              / COUNT(*), 2)                                                           AS pct_relleno
    FROM bronze.ventas_minoristas
    WHERE fecha_venta BETWEEN DATE '2022-01-01' AND DATE '2025-12-31'
    GROUP BY id_tipo_pedido, desc_tipo_pedido
    ORDER BY filas DESC
""").fetchdf()
print(por_canal.to_string(index=False))

# 5. Comparación entre las dos columnas de importe (¿coinciden? ¿hay descuentos aplicados?)
print("\n--- 5. COMPARATIVA importe_total vs importe_total_con_descuento ---")
comparativa = con.execute("""
    SELECT
        COUNT(*)                                                                       AS total_filas_con_ambos,
        SUM(CASE WHEN importe_total = importe_total_con_descuento THEN 1 ELSE 0 END)   AS coinciden,
        SUM(CASE WHEN importe_total > importe_total_con_descuento THEN 1 ELSE 0 END)   AS hay_descuento,
        SUM(CASE WHEN importe_total < importe_total_con_descuento THEN 1 ELSE 0 END)   AS imp_desc_mayor,
        ROUND(AVG(importe_total - importe_total_con_descuento), 2)                     AS dto_medio_eur
    FROM bronze.ventas_minoristas
    WHERE importe_total IS NOT NULL
      AND importe_total_con_descuento IS NOT NULL
""").fetchdf()
print(comparativa.T)

ANÁLISIS DE LAS NUEVAS COLUMNAS DE IMPORTE

--- 1. COBERTURA GENERAL (sobre el bronze completo) ---
                           0
total_filas        4009222.0
imp_total_nulos          0.0
imp_total_validos  4009222.0
imp_desc_nulos           0.0
imp_desc_validos   4009222.0

--- 2. ESTADÍSTICAS DESCRIPTIVAS (filas no nulas) ---
                    columna  no_nulos   minimo  media  mediana  maximo  negativos    ceros
              importe_total   4009222 -34696.6  27.44    20.00 16888.8   159770.0 103751.0
importe_total_con_descuento   4009222 -34696.6  27.34    19.95 16888.8   159613.0 109913.0

--- 3. COBERTURA POR AÑO (ventana 2022-2025) ---
 anio  filas  con_importe  pct_relleno
 2022 729665     729665.0        100.0
 2023 719482     719482.0        100.0
 2024 705208     705208.0        100.0
 2025 713210     713210.0        100.0

--- 4. COBERTURA POR CANAL (id_tipo_pedido) ---
id_tipo_pedido   desc_tipo_pedido   filas  con_importe  pct_relleno
             1          Temporada 16

## 3. Construcción de `silver.ventas_minoristas`

A partir de los hallazgos de la fase de inspección, se aplican las siguientes transformaciones para generar la tabla Silver definitiva:

| Transformación | Detalle |
|---|---|
| **Filtro temporal** | Mantener únicamente registros con `fecha_venta` entre 2022-01-01 y 2025-12-31. El CSV de origen contiene datos de 2019 a 2026, por lo que este filtro aplicado en Silver descarta aproximadamente 1,14 millones de registros fuera de la ventana del proyecto. |
| **Casteo de tipos** | `id_cliente` se castea de BIGINT a VARCHAR para garantizar la coherencia referencial con `silver.dim_cliente` y `silver.fact_lineas_pedido`. |
| **Normalización de texto** | Aplicación de `UPPER(TRIM(...))` sobre los códigos identificativos (`cod_modelo`, `cod_color`, `cod_serie`, `talla`, `id_tipo_pedido`) y de `TRIM(...)` sobre las descripciones textuales. |
| **Identificador de producto** | Construcción del campo `id_sku` mediante la concatenación normalizada de modelo, color y talla, separados por guiones. |
| **Flag de devolución** | Cálculo del campo booleano `es_devolucion` a partir del listado de códigos de tipo de pedido proporcionado por la empresa (códigos 3, 10, 12, D77/88, EDEV, DPER y ZCR), garantizando una clasificación independiente del signo del importe. |
| **Incorporación de importes** | Incorporación directa de las columnas `importe_total` e `importe_total_con_descuento` desde la capa Bronze, manteniendo el tipo DOUBLE y la totalidad de los valores. |
| **Campo derivado de descuento** | Generación de la columna `descuento_aplicado`, calculada como la diferencia entre el importe bruto y el importe con descuento. Los valores positivos indican descuentos efectivamente aplicados, los valores cero indican ausencia de descuento y los valores ligeramente negativos (0,17 % de las filas) corresponden a inconsistencias residuales del ERP que se documentan pero no se corrigen. |

In [38]:
print("Construyendo silver.ventas_minoristas con importes... (puede tardar 30-60 s)\n")

con.execute("""
    CREATE OR REPLACE TABLE silver.ventas_minoristas AS
    SELECT
        -- Fecha
        fecha_venta,

        -- Cliente (casteado a VARCHAR para coherencia entre tablas)
        CAST(id_cliente AS VARCHAR) AS id_cliente,

        -- Producto: códigos normalizados
        UPPER(TRIM(cod_modelo))  AS cod_modelo,
        UPPER(TRIM(cod_color))   AS cod_color,
        UPPER(TRIM(cod_serie))   AS cod_serie,
        UPPER(TRIM(talla))       AS talla,

        -- Producto: descripciones (solo TRIM, conservan capitalización original)
        TRIM(desc_modelo) AS desc_modelo,
        TRIM(desc_color)  AS desc_color,
        TRIM(desc_serie)  AS desc_serie,

        -- SKU derivado
        UPPER(TRIM(cod_modelo)) || '-' || UPPER(TRIM(cod_color)) || '-' || UPPER(TRIM(talla)) AS id_sku,

        -- Tipo de pedido (canal)
        UPPER(TRIM(id_tipo_pedido)) AS id_tipo_pedido,
        TRIM(desc_tipo_pedido)      AS desc_tipo_pedido,

        -- Flag de devolución (basado en tipo de pedido, no en signo del importe)
        CASE 
            WHEN UPPER(TRIM(id_tipo_pedido)) IN ('3', '10', '12', 'D77/88', 'EDEV', 'DPER', 'ZCR') 
            THEN TRUE 
            ELSE FALSE 
        END AS es_devolucion,

        -- Cantidades
        cantidad_neta,
        cantidad_ventas_sin_devoluciones,

        -- Importes económicos (recibidos del ERP en la actualización pendiente)
        importe_total,
        importe_total_con_descuento,

        -- Descuento aplicado: diferencia entre importe bruto y neto
        -- Positivo = se aplicó descuento; cero = sin descuento; negativo = inconsistencia ERP (0,17 % de las filas)
        ROUND(importe_total - importe_total_con_descuento, 2) AS descuento_aplicado,

        -- Temporada
        id_temporada

    FROM bronze.ventas_minoristas
    WHERE fecha_venta BETWEEN DATE '2022-01-01' AND DATE '2025-12-31'
""")

print("✅ silver.ventas_minoristas creada correctamente con los importes incorporados")

Construyendo silver.ventas_minoristas con importes... (puede tardar 30-60 s)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ silver.ventas_minoristas creada correctamente con los importes incorporados


## 4. Validación de la tabla resultante

In [39]:
print("=" * 60)
print("AUDITORÍA DE VOLÚMENES: BRONZE → SILVER")
print("=" * 60)

n_bronze = con.execute("SELECT COUNT(*) FROM bronze.ventas_minoristas").fetchone()[0]
n_silver = con.execute("SELECT COUNT(*) FROM silver.ventas_minoristas").fetchone()[0]

print(f"Filas en bronze : {n_bronze:>12,}")
print(f"Filas en silver : {n_silver:>12,}")
print(f"Diferencia      : {n_bronze - n_silver:>12,}")
print(f"% conservado    : {n_silver / n_bronze * 100:>12.2f} %")

AUDITORÍA DE VOLÚMENES: BRONZE → SILVER
Filas en bronze :    4,009,222
Filas en silver :    2,867,565
Diferencia      :    1,141,657
% conservado    :        71.52 %


In [40]:
print("=" * 60)
print("ESQUEMA DE silver.ventas_minoristas")
print("=" * 60)
esquema_silver = con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver' AND table_name = 'ventas_minoristas'
    ORDER BY ordinal_position
""").fetchdf()
print(esquema_silver.to_string(index=False))

ESQUEMA DE silver.ventas_minoristas
                     column_name data_type
                     fecha_venta      DATE
                      id_cliente   VARCHAR
                      cod_modelo   VARCHAR
                       cod_color   VARCHAR
                       cod_serie   VARCHAR
                           talla   VARCHAR
                     desc_modelo   VARCHAR
                      desc_color   VARCHAR
                      desc_serie   VARCHAR
                          id_sku   VARCHAR
                  id_tipo_pedido   VARCHAR
                desc_tipo_pedido   VARCHAR
                   es_devolucion   BOOLEAN
                   cantidad_neta    BIGINT
cantidad_ventas_sin_devoluciones    BIGINT
                   importe_total    DOUBLE
     importe_total_con_descuento    DOUBLE
              descuento_aplicado    DOUBLE
                    id_temporada    BIGINT


In [41]:
print("=" * 60)
print("DISTRIBUCIÓN TEMPORAL FINAL (silver)")
print("=" * 60)
temp = con.execute("""
    SELECT 
        EXTRACT(YEAR FROM fecha_venta) AS anio,
        COUNT(*) AS filas,
        COUNT(DISTINCT id_cliente) AS clientes,
        COUNT(DISTINCT id_sku) AS skus_distintos,
        SUM(cantidad_neta) AS suma_cantidad_neta
    FROM silver.ventas_minoristas
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(temp.to_string(index=False))

DISTRIBUCIÓN TEMPORAL FINAL (silver)
 anio  filas  clientes  skus_distintos  suma_cantidad_neta
 2022 729665      2266           27839           1256163.0
 2023 719482      2232           24396           1186289.0
 2024 705208      2200           21466           1119199.0
 2025 713210      2117           24446           1107413.0


In [42]:
print("=" * 60)
print("VALIDACIÓN DEL FLAG es_devolucion")
print("=" * 60)
val_dev = con.execute("""
    SELECT 
        es_devolucion,
        COUNT(*) AS filas,
        SUM(cantidad_neta) AS suma_cantidad,
        ROUND(AVG(cantidad_neta), 2) AS media_cantidad
    FROM silver.ventas_minoristas
    GROUP BY es_devolucion
    ORDER BY es_devolucion
""").fetchdf()
print(val_dev.to_string(index=False))

VALIDACIÓN DEL FLAG es_devolucion
 es_devolucion   filas  suma_cantidad  media_cantidad
         False 2708702      4873524.0            1.80
          True  158863      -204460.0           -1.29


In [43]:
print("=" * 60)
print("INTEGRIDAD REFERENCIAL CON silver.dim_cliente")
print("=" * 60)
integ = con.execute("""
    SELECT 
        COUNT(DISTINCT v.id_cliente) AS clientes_en_ventas,
        COUNT(DISTINCT d.id_cliente) AS clientes_en_dim,
        COUNT(DISTINCT CASE WHEN d.id_cliente IS NULL THEN v.id_cliente END) AS clientes_huerfanos
    FROM silver.ventas_minoristas v
    LEFT JOIN silver.dim_cliente d ON v.id_cliente = d.id_cliente
""").fetchdf()
print(integ.to_string(index=False))

INTEGRIDAD REFERENCIAL CON silver.dim_cliente
 clientes_en_ventas  clientes_en_dim  clientes_huerfanos
               3055             3054                   1


In [44]:
print("=" * 60)
print("TOP 10 SKUs POR CANTIDAD VENDIDA (excluyendo devoluciones)")
print("=" * 60)
top_skus = con.execute("""
    SELECT 
        id_sku,
        desc_modelo,
        desc_color,
        talla,
        SUM(cantidad_neta) AS unidades_vendidas
    FROM silver.ventas_minoristas
    WHERE es_devolucion = FALSE
    GROUP BY id_sku, desc_modelo, desc_color, talla
    ORDER BY unidades_vendidas DESC
    LIMIT 10
""").fetchdf()
print(top_skus.to_string(index=False))

TOP 10 SKUs POR CANTIDAD VENDIDA (excluyendo devoluciones)
      id_sku                   desc_modelo desc_color talla  unidades_vendidas
 10630-035-M       SUJETADOR SIN AROS 24-H MAQUILLAJE     M            14916.0
002008-185-M      CINTA LIFT-TAPE ADHESIVA       PIEL     M            14605.0
 10630-035-L       SUJETADOR SIN AROS 24-H MAQUILLAJE     L            13804.0
 10615-035-M SUJETADOR ESCOTE PROFUNDO 24H MAQUILLAJE     M            12124.0
 10630-004-M       SUJETADOR SIN AROS 24-H      NEGRO     M            11662.0
 10630-004-L       SUJETADOR SIN AROS 24-H      NEGRO     L            10943.0
10630-035-XL       SUJETADOR SIN AROS 24-H MAQUILLAJE    XL            10493.0
 10615-035-L SUJETADOR ESCOTE PROFUNDO 24H MAQUILLAJE     L             9621.0
 10630-035-S       SUJETADOR SIN AROS 24-H MAQUILLAJE     S             9402.0
 10615-004-M SUJETADOR ESCOTE PROFUNDO 24H      NEGRO     M             9053.0


In [45]:
print("=" * 60)
print("VALIDACIÓN DE LOS IMPORTES EN silver.ventas_minoristas")
print("=" * 60)

# 1. Cobertura final
print("\n--- 1. COBERTURA DE LOS IMPORTES EN SILVER ---")
cobertura_silver = con.execute("""
    SELECT
        COUNT(*)                                                                   AS total_filas,
        SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END)                 AS imp_total_validos,
        SUM(CASE WHEN importe_total_con_descuento IS NOT NULL THEN 1 ELSE 0 END)   AS imp_desc_validos,
        ROUND(100.0 * SUM(CASE WHEN importe_total IS NOT NULL THEN 1 ELSE 0 END) 
              / COUNT(*), 2)                                                       AS pct_cobertura
    FROM silver.ventas_minoristas
""").fetchdf()
print(cobertura_silver.T)

# 2. Descomposición por flag de devolución
print("\n--- 2. IMPORTE POR FLAG DE DEVOLUCIÓN ---")
por_devolucion = con.execute("""
    SELECT
        es_devolucion,
        COUNT(*)                                  AS num_filas,
        ROUND(SUM(importe_total), 2)              AS suma_importe_total,
        ROUND(SUM(importe_total_con_descuento), 2) AS suma_importe_neto,
        ROUND(MIN(importe_total), 2)              AS min_importe,
        ROUND(MAX(importe_total), 2)              AS max_importe,
        ROUND(MEDIAN(importe_total), 2)           AS mediana_importe
    FROM silver.ventas_minoristas
    GROUP BY es_devolucion
    ORDER BY es_devolucion
""").fetchdf()
print(por_devolucion.to_string(index=False))

# 3. Resumen económico global del periodo (excluyendo devoluciones)
print("\n--- 3. RESUMEN ECONÓMICO DEL CANAL RETAIL 2022-2025 ---")
resumen = con.execute("""
    SELECT
        COUNT(*)                                          AS num_operaciones_venta,
        ROUND(SUM(importe_total_con_descuento), 2)        AS facturacion_neta_eur,
        ROUND(AVG(importe_total_con_descuento), 2)        AS importe_medio_eur,
        ROUND(MEDIAN(importe_total_con_descuento), 2)     AS importe_mediana_eur
    FROM silver.ventas_minoristas
    WHERE es_devolucion = FALSE
""").fetchdf()
print(resumen.T)

# 4. Descomposición anual de la facturación retail
print("\n--- 4. FACTURACIÓN RETAIL POR AÑO (sin devoluciones) ---")
por_anio = con.execute("""
    SELECT
        EXTRACT(YEAR FROM fecha_venta)                    AS anio,
        COUNT(*)                                          AS num_operaciones,
        ROUND(SUM(importe_total_con_descuento), 2)        AS facturacion_eur,
        ROUND(AVG(importe_total_con_descuento), 2)        AS ticket_medio_eur
    FROM silver.ventas_minoristas
    WHERE es_devolucion = FALSE
    GROUP BY anio
    ORDER BY anio
""").fetchdf()
print(por_anio.to_string(index=False))

# 5. Filas anómalas: importe_total_con_descuento > importe_total (inconsistencia)
print("\n--- 5. FILAS ANÓMALAS (importe_con_descuento > importe_total) ---")
anomalas = con.execute("""
    SELECT COUNT(*) AS filas_anomalas
    FROM silver.ventas_minoristas
    WHERE importe_total_con_descuento > importe_total
""").fetchone()[0]
print(f"Filas con importe_con_descuento mayor que importe_total: {anomalas:,}")
print("(Inconsistencias residuales del ERP, ~0,17 % del total. Se documentan pero no se corrigen.)")

VALIDACIÓN DE LOS IMPORTES EN silver.ventas_minoristas

--- 1. COBERTURA DE LOS IMPORTES EN SILVER ---
                           0
total_filas        2867565.0
imp_total_validos  2867565.0
imp_desc_validos   2867565.0
pct_cobertura          100.0

--- 2. IMPORTE POR FLAG DE DEVOLUCIÓN ---
 es_devolucion  num_filas  suma_importe_total  suma_importe_neto  min_importe  max_importe  mediana_importe
         False    2708702         85187086.82        84945543.37      -1359.2     16888.80            20.45
          True     158863         -4476548.19        -4444068.09     -34696.6       494.16           -23.10

--- 3. RESUMEN ECONÓMICO DEL CANAL RETAIL 2022-2025 ---
                                 0
num_operaciones_venta   2708702.00
facturacion_neta_eur   84945543.37
importe_medio_eur            31.36
importe_mediana_eur          20.45

--- 4. FACTURACIÓN RETAIL POR AÑO (sin devoluciones) ---
 anio  num_operaciones  facturacion_eur  ticket_medio_eur
 2022           694441      21276251.

In [46]:
print("=" * 60)
print("INVESTIGACIÓN DE LOS 3 CLIENTES HUÉRFANOS")
print("=" * 60)
huerfanos = con.execute("""
    SELECT 
        v.id_cliente,
        COUNT(*) AS filas_ventas,
        SUM(v.cantidad_neta) AS unidades,
        MIN(v.fecha_venta) AS primera_venta,
        MAX(v.fecha_venta) AS ultima_venta,
        COUNT(DISTINCT v.desc_tipo_pedido) AS canales_distintos,
        STRING_AGG(DISTINCT v.desc_tipo_pedido, ', ') AS canales
    FROM silver.ventas_minoristas v
    LEFT JOIN silver.dim_cliente d ON v.id_cliente = d.id_cliente
    WHERE d.id_cliente IS NULL
    GROUP BY v.id_cliente
    ORDER BY filas_ventas DESC
""").fetchdf()
print(huerfanos.to_string(index=False))

INVESTIGACIÓN DE LOS 3 CLIENTES HUÉRFANOS
id_cliente  filas_ventas  unidades primera_venta ultima_venta  canales_distintos   canales
     32437             1       1.0    2025-11-07   2025-11-07                  1 Temporada


## 5. Conclusiones del notebook 05

### Resumen del proceso

La tabla `silver.ventas_minoristas` queda construida sobre los 2,87 millones de operaciones registradas en el canal retail durante el cuatrienio 2022-2025, a partir de los 4.009.222 registros de la capa Bronze. La reducción de volumen corresponde íntegramente a la aplicación del filtro temporal de la ventana del proyecto (descarte de los datos de 2019 a 2021 e inicio de 2026).

La construcción ha aplicado las transformaciones técnicas previstas: normalización en mayúsculas y sin espacios de los códigos de modelo, color, serie y talla; construcción del identificador derivado `id_sku` mediante la concatenación de modelo, color y talla; armonización del identificador de cliente con la dimensión maestra; e incorporación del flag booleano `es_devolucion`, calculado a partir del catálogo de códigos de tipo de pedido proporcionado por la empresa (códigos 3, 10, 12, D77/88, EDEV, DPER y ZCR).

### Incorporación de los importes económicos

La actualización del sistema ERP ha permitido incorporar a la tabla las dos columnas de importe (`importe_total` e `importe_total_con_descuento`) con una cobertura del 100 % sobre la totalidad de los registros y con valores numéricos válidos en todos los canales y años de la ventana de análisis. Adicionalmente, se ha generado el campo derivado `descuento_aplicado`, calculado como la diferencia entre el importe bruto y el neto, que permite cuantificar de forma directa el descuento aplicado a cada operación.

El análisis de cobertura revela que el descuento real aplicado en el canal retail es marginal: el 97,9 % de las operaciones presenta importes idénticos en ambas columnas (descuento cero) y el descuento medio agregado se sitúa en 0,11 euros por operación. Únicamente el 1,95 % de las operaciones presenta un descuento estrictamente positivo. Un residuo del 0,17 % de las filas presenta una inconsistencia menor (importe con descuento mayor que importe bruto) atribuible a anomalías de carga del ERP que no comprometen la fiabilidad agregada de los datos.

### Hallazgos relevantes para los análisis posteriores

La descomposición de los importes por flag de devolución confirma la coherencia interna de los datos: los movimientos marcados como devolución presentan importes negativos sistemáticos, mientras que las operaciones de venta presentan importes positivos. Esta separación, junto con la disponibilidad del campo `es_devolucion`, permitirá calcular de forma robusta la facturación neta retail en el bloque de comportamiento de la tabla `gold.cliente_360`.

La integridad referencial con `silver.dim_cliente` se ha validado satisfactoriamente. Los clientes huérfanos identificados (sin registro en la dimensión maestra) presentan un volumen marginal y se documentan caso a caso para su tratamiento en la capa Gold.

### Próximo paso

Con esta tabla cerrada, queda desbloqueada la activación de los Bloques 4 (Comportamiento) y 5 (Temporal) de la tabla `gold.cliente_360`. La activación se realizará mediante el cambio de la bandera `EJECUTAR_BLOQUES_RETAIL` del notebook 08 a valor `True` y su re-ejecución completa.

In [47]:
con.close()
print("✅ Conexión cerrada. silver.ventas_minoristas guardada en disco.")

✅ Conexión cerrada. silver.ventas_minoristas guardada en disco.
